# Section 5: Loss Functions (Softmax and Cross-Entropy)

In the previous section, we built a Linear Classifier capable of generating 10 arbitrary scores (logits) for any given input image. 

However, when we initialize our weight matrix $\mathbf{W}$ randomly, these scores will be completely meaningless. To train the model, we need a mathematical way to quantify how "unhappy" we are with the current scores. This measurement is called the **Loss Function** (or Objective Function). Our ultimate goal in machine learning is to *minimize* this loss.

## 5.1 The Need for Probabilities

Let's say our linear classifier produces the following raw scores for an image of a cat:
*   Cat: 3.2
*   Dog: 5.1
*   Ship: -1.7

These numbers are unbounded and difficult to interpret. It thinks "Dog" is the highest, but what does 5.1 mean in absolute terms? To make these scores mathematically manageable and interpretable, we want to convert them into a **probability distribution** over the classes. 

A valid probability distribution must satisfy two rules:
1.  All values must be positive ($P \geq 0$).
2.  All values must sum to 1 ($\sum P = 1$).

## 5.2 The Softmax Function

The **Softmax function** takes a vector of arbitrary real-valued scores (often called *logits*) and squashes them into a valid probability distribution.

Let $s_k$ be the raw score for class $k$. The probability assigned to class $k$ is given by:

$$ P(Y=k | X=x_i) = \frac{e^{s_k}}{\sum_{j} e^{s_j}} $$

**Step-by-step breakdown:**
1.  **Exponentiate:** We take $e^{s_k}$ for every score. Because $e^x$ is always positive, this guarantees all our numbers are now strictly positive, satisfying the first rule of probabilities. It also heavily penalizes negative scores and amplifies positive scores.
2.  **Normalize:** We divide each exponentiated score by the sum of all exponentiated scores. This guarantees that the final outputs sum to 1, satisfying the second rule.

If we apply Softmax to our earlier example:
*   $e^{3.2} \approx 24.5$
*   $e^{5.1} \approx 164.0$
*   $e^{-1.7} \approx 0.18$
*   **Sum** $\approx 188.68$

Normalized probabilities:
*   $P(\text{Cat}) = 24.5 / 188.68 \approx 0.13$
*   $P(\text{Dog}) = 164.0 / 188.68 \approx 0.87$
*   $P(\text{Ship}) = 0.18 / 188.68 \approx 0.001$

The model is 13% confident the image is a cat, and 87% confident it's a dog.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# --- Numpy Softmax ---
scores = np.array([3.2, 5.1, -1.7])

# Step 1: Exponentiate
exp_scores = np.exp(scores)

# Step 2: Normalize
probabilities = exp_scores / np.sum(exp_scores)

print("--- NUMPY ---")
print(f"Raw scores: {scores}")
print(f"Probabilities: {probabilities}")
print(f"Sum of probabilities: {np.sum(probabilities):.2f}\n")

# --- PyTorch Softmax ---
scores_torch = torch.tensor([3.2, 5.1, -1.7])
# dim=0 because this is just a 1D tensor
probs_torch = F.softmax(scores_torch, dim=0) 

print("--- PYTORCH ---")
print(f"PyTorch Probabilities: {probs_torch}")

## 5.3 Cross-Entropy Loss

Now that we have probabilities, how do we define the loss? We want the probability of the *correct* class to be as close to 1 as possible (and consequently, all incorrect classes close to 0).

Let $y_i$ be the true correct class label. We want to **maximize** $P(Y=y_i | X=x_i)$.

In optimization, we conventionally *minimize* loss. To turn a maximization problem into a minimization problem, we negate it. Furthermore, taking the natural logarithm ($\log$) of the probability makes the math vastly easier (especially for computing gradients later) without changing the location of the minimum.

Thus, we arrive at the **Cross-Entropy Loss** $L_i$ for a single example:

$$ L_i = -\log \left( \frac{e^{s_{y_i}}}{\sum_{j} e^{s_j}} \right) $$

Or more simply:
$$ L_i = -\log(P_{y_i}) $$

Where $P_{y_i}$ is the Softmax probability assigned to the correct class.

> [!NOTE]
> If the model is 100% confident in the correct class ($P = 1$), the loss is $-\log(1) = 0$. If it is completely wrong ($P \to 0$), the loss approaches $\infty$.

## 5.4 Information Theory Perspective

Why is it called "Cross-Entropy"? In Information Theory, the **Kullback-Leibler (KL) Divergence** measures how different a predicted probability distribution $Q$ is from a true target distribution $P$.

Our true distribution $P$ is a "one-hot" vector (all probability mass is on the correct class, e.g., `[1, 0, 0]`). Our predicted distribution $Q$ is the output of the Softmax function.

Minimizing the KL Divergence between the one-hot target distribution and our predicted distribution mathematically simplifies down to the exact same formula: $-\log(P_{y_i})$. Thus, Cross-Entropy Loss is equivalent to forcing our predicted distribution to match the ground-truth one-hot distribution.

## 5.5 Loss Characteristics and Sanity Checks

Understanding the mathematical bounds of your loss function is critical for debugging neural networks.

### Min and Max Values
*   **Minimum Loss:** $0$. Occurs when the probability of the correct class is exactly $1.0$.
*   **Maximum Loss:** $\infty$. Occurs when the probability of the correct class is exactly $0.0$.

### The Initialization Sanity Check
When you first initialize a linear classifier with small random weights, all scores $s_j$ will be roughly equal, hovering near $0$. 
If all scores are equal, the Softmax probabilities will be roughly uniform. If you have $C$ classes, the probability of the correct class will be $P_{y_i} \approx \frac{1}{C}$.

Therefore, your loss at step 0 (initialization) should always be:
$$ L \approx -\log\left(\frac{1}{C}\right) = \ln(C) $$

For CIFAR-10 ($C=10$), the expected initial loss is $\ln(10) \approx 2.3$.

**Debugging Tip:** If you build a neural network for CIFAR-10 and your initial loss is $15.0$ or $0.1$, you know you have a bug in your code before you even start training!

In [ ]:
import numpy as np

C = 10 # CIFAR-10 classes
expected_loss = -np.log(1/C)
print(f"Expected initial loss for {C} classes: {expected_loss:.4f}")

# Let's verify with code!
# Small random weights ~ 0
W_init = np.random.randn(10, 3072) * 0.0001 
x_dummy = np.random.randn(3072, 1)

scores_init = W_init.dot(x_dummy).flatten()
print(f"Initial raw scores: {scores_init[:4]}...") # Since weights are tiny, scores are all ~0

# Softmax
exp_scores_init = np.exp(scores_init)
probs_init = exp_scores_init / np.sum(exp_scores_init)

# Assume the correct class is randomly index 3
correct_class = 3
p_correct = probs_init[correct_class]

initial_loss = -np.log(p_correct)
print(f"Calculated initial loss: {initial_loss:.4f}")

---
### Summary of Section 5
*   **Concepts Introduced:** Loss/Objective functions, Softmax function, Cross-Entropy Loss, Logits to Probabilities, KL Divergence equivalence, Initialization Sanity Checks.
*   **Equations Derived:** 
    *   Softmax: $P(Y=k | X=x_i) = \frac{e^{s_k}}{\sum_{j} e^{s_j}}$
    *   Cross-Entropy: $L_i = -\log(P_{y_i})$
*   **Notation Introduced:** $L_i$ (Loss for a single example), $s_k$ (Logit score for class $k$), $P_{y_i}$ (Probability of correct class).
*   **Conclusion:** We have successfully built the architecture to generate predictions and evaluate those predictions mathematically. The final remaining piece (covered in the next lecture) is **Optimization**: computing gradients to update $\mathbf{W}$ and actually minimize this loss!